In [ ]:
# CELL 1 — Imports and initialization

import cv2
import time
import math
import heapq
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from jetbot import Robot, Camera, bgr8_to_jpeg
from src.SCSCtrl import TTLServo

robot = Robot()
camera = Camera.instance(width=320, height=240)

debug_image = widgets.Image(format='jpeg', width=640, height=480)
display(debug_image)

def hard_stop():
    robot.stop()

In [ ]:
# CELL 1 — Imports and initialization

import cv2
import time
import math
import heapq
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from jetbot import Robot, Camera, bgr8_to_jpeg
from src.SCSCtrl import TTLServo

robot = Robot()
camera = Camera.instance(width=320, height=240)

debug_image = widgets.Image(format='jpeg', width=640, height=480)
display(debug_image)

def hard_stop():
    robot.stop()

In [ ]:
# CELL 3 — Utility functions

def hsv_mask(frame, lower, upper):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array(lower), np.array(upper))
    mask = cv2.erode(mask, None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)
    return mask

def clean_mask(mask, k=5):
    kernel = np.ones((k, k), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

def contour_center(cnt):
    M = cv2.moments(cnt)
    if M["m00"] == 0:
        return None
    return int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])

def image_to_world(px, py):
    p = np.array([[[px, py]]], dtype=np.float32)
    out = cv2.perspectiveTransform(p, H_img_to_world)[0][0]
    return float(out[0]), float(out[1])

def world_to_image(x, y):
    p = np.array([[[x, y]]], dtype=np.float32)
    out = cv2.perspectiveTransform(p, H_world_to_img)[0][0]
    return int(out[0]), int(out[1])

def distance(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

In [ ]:
# CELL 4 — Object detection

def detect_colored_objects(frame, hsv_names, min_area=80, max_area=999999):
    detections = []

    for name in hsv_names:
        lower, upper = CFG["HSV"][name]
        mask = clean_mask(hsv_mask(frame, lower, upper))
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in cnts:
            area = cv2.contourArea(cnt)
            if area < min_area or area > max_area:
                continue

            center = contour_center(cnt)
            if center is None:
                continue

            x, y, w, h = cv2.boundingRect(cnt)
            circularity = 0
            perimeter = cv2.arcLength(cnt, True)
            if perimeter > 0:
                circularity = 4 * math.pi * area / (perimeter * perimeter)

            wx, wy = image_to_world(center[0], center[1])

            detections.append({
                "type": name,
                "center_px": center,
                "bbox": (x, y, w, h),
                "area": area,
                "circularity": circularity,
                "world": (wx, wy),
                "contour": cnt
            })

    return detections

def detect_boundary(frame):
    mask = clean_mask(hsv_mask(frame, *CFG["HSV"]["yellow"]))
    edges = cv2.Canny(mask, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=35,
                            minLineLength=40, maxLineGap=15)
    return mask, lines

def detect_obstacles(frame):
    blue = detect_colored_objects(frame, ["blue"], CFG["min_obstacle_area"])
    green = detect_colored_objects(frame, ["green"], CFG["min_obstacle_area"])
    for o in blue:
        o["kind"] = "obstacle_blue"
    for o in green:
        o["kind"] = "obstacle_green"
    return blue + green

def detect_grey_box(frame):
    boxes = detect_colored_objects(frame, ["grey"], CFG["min_box_area"])
    for b in boxes:
        b["kind"] = "drop_box"
    if not boxes:
        return None
    return max(boxes, key=lambda x: x["area"])

def looks_like_sphere(det):
    # Spheres usually look more circular and larger/taller.
    return det["circularity"] > 0.72 and det["area"] >= CFG["min_sphere_area"]

def detect_spheres(frame):
    spheres = detect_colored_objects(
        frame,
        ["sphere_orange", "sphere_purple"],
        CFG["min_sphere_area"]
    )
    for s in spheres:
        s["kind"] = "sphere"
    return spheres

def detect_bottle_caps(frame):
    raw_caps = detect_colored_objects(
        frame,
        ["cap_red_1", "cap_red_2", "cap_white"],
        CFG["min_cap_area"],
        CFG["max_cap_area"]
    )

    caps = []
    for c in raw_caps:
        # Bottle caps are usually flatter: reject very circular/larger sphere-like objects.
        if looks_like_sphere(c):
            continue

        x, y, w, h = c["bbox"]
        aspect = w / max(h, 1)

        if 0.45 <= aspect <= 2.2:
            c["kind"] = "bottle_cap"
            caps.append(c)

    return caps

In [ ]:
# CELL 5 — Mapping with inflated forbidden zones

def make_empty_grid():
    gw = int(CFG["field_w_m"] / CFG["grid_res_m"])
    gh = int(CFG["field_h_m"] / CFG["grid_res_m"])
    grid = np.zeros((gh, gw), dtype=np.uint8)
    return grid

def world_to_grid(x, y):
    gx = int(x / CFG["grid_res_m"])
    gy = int(y / CFG["grid_res_m"])
    return gx, gy

def grid_to_world(gx, gy):
    return (
        (gx + 0.5) * CFG["grid_res_m"],
        (gy + 0.5) * CFG["grid_res_m"]
    )

def mark_circle(grid, center, radius_m, value=1):
    cx, cy = world_to_grid(center[0], center[1])
    r = int(radius_m / CFG["grid_res_m"])
    h, w = grid.shape

    for yy in range(max(0, cy-r), min(h, cy+r+1)):
        for xx in range(max(0, cx-r), min(w, cx+r+1)):
            wx, wy = grid_to_world(xx, yy)
            if distance((wx, wy), center) <= radius_m:
                grid[yy, xx] = value

def build_local_map(obstacles):
    grid = make_empty_grid()
    h, w = grid.shape

    body_radius = max(
        CFG["robot_width_m"] / 2,
        CFG["robot_length_m"] / 2 + CFG["front_claw_overhang_m"]
    )

    # Boundary safety margin
    margin_cells = int(CFG["boundary_margin_m"] / CFG["grid_res_m"])
    grid[:margin_cells, :] = 1
    grid[-margin_cells:, :] = 1
    grid[:, :margin_cells] = 1
    grid[:, -margin_cells:] = 1

    # Inflated obstacle zones
    inflated_radius = body_radius + CFG["obstacle_margin_m"]

    for obs in obstacles:
        mark_circle(grid, obs["world"], inflated_radius, 1)

    return grid

In [ ]:
# CELL 6 — A* path planning

def astar(grid, start_w, goal_w):
    start = world_to_grid(*start_w)
    goal = world_to_grid(*goal_w)

    h, w = grid.shape

    def valid(node):
        x, y = node
        return 0 <= x < w and 0 <= y < h and grid[y, x] == 0

    if not valid(start) or not valid(goal):
        return None

    def heuristic(a, b):
        return math.hypot(a[0]-b[0], a[1]-b[1])

    neighbors = [
        (1,0), (-1,0), (0,1), (0,-1),
        (1,1), (1,-1), (-1,1), (-1,-1)
    ]

    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g = {start: 0}

    while open_set:
        _, current = heapq.heappop(open_set)

        if current == goal:
            path = []
            while current in came_from:
                path.append(grid_to_world(*current))
                current = came_from[current]
            path.append(grid_to_world(*start))
            path.reverse()
            return path

        for dx, dy in neighbors:
            nxt = (current[0] + dx, current[1] + dy)
            if not valid(nxt):
                continue

            step_cost = math.hypot(dx, dy)
            tentative = g[current] + step_cost

            if nxt not in g or tentative < g[nxt]:
                came_from[nxt] = current
                g[nxt] = tentative
                f = tentative + heuristic(nxt, goal)
                heapq.heappush(open_set, (f, nxt))

    return None

def simplify_path(path, step=3):
    if not path:
        return None
    simple = path[::step]
    if simple[-1] != path[-1]:
        simple.append(path[-1])
    return simple

In [ ]:
# CELL 6 — A* path planning

def astar(grid, start_w, goal_w):
    start = world_to_grid(*start_w)
    goal = world_to_grid(*goal_w)

    h, w = grid.shape

    def valid(node):
        x, y = node
        return 0 <= x < w and 0 <= y < h and grid[y, x] == 0

    if not valid(start) or not valid(goal):
        return None

    def heuristic(a, b):
        return math.hypot(a[0]-b[0], a[1]-b[1])

    neighbors = [
        (1,0), (-1,0), (0,1), (0,-1),
        (1,1), (1,-1), (-1,1), (-1,-1)
    ]

    open_set = []
    heapq.heappush(open_set, (0, start))
    came_from = {}
    g = {start: 0}

    while open_set:
        _, current = heapq.heappop(open_set)

        if current == goal:
            path = []
            while current in came_from:
                path.append(grid_to_world(*current))
                current = came_from[current]
            path.append(grid_to_world(*start))
            path.reverse()
            return path

        for dx, dy in neighbors:
            nxt = (current[0] + dx, current[1] + dy)
            if not valid(nxt):
                continue

            step_cost = math.hypot(dx, dy)
            tentative = g[current] + step_cost

            if nxt not in g or tentative < g[nxt]:
                came_from[nxt] = current
                g[nxt] = tentative
                f = tentative + heuristic(nxt, goal)
                heapq.heappush(open_set, (f, nxt))

    return None

def simplify_path(path, step=3):
    if not path:
        return None
    simple = path[::step]
    if simple[-1] != path[-1]:
        simple.append(path[-1])
    return simple

In [ ]:
# CELL 8 — Claw control

# Servo IDs based on Waveshare examples:
# Servo 4 = claw open/close
# xyInput controls arm X/Y position
# Tune these positions carefully.

CLAW_OPEN_ANGLE = -10
CLAW_CLOSED_ANGLE = -75

ARM_TRAVEL_X = 130
ARM_TRAVEL_Y = 20

ARM_PICK_X = 150
ARM_PICK_Y = -45

ARM_LIFT_X = 120
ARM_LIFT_Y = 45

ARM_DROP_X = 160
ARM_DROP_Y = -20

def servo_delay():
    time.sleep(0.25)

def claw_open():
    TTLServo.servoAngleCtrl(4, CLAW_OPEN_ANGLE, 1, 150)
    servo_delay()

def claw_close():
    TTLServo.servoAngleCtrl(4, CLAW_CLOSED_ANGLE, 1, 150)
    servo_delay()

def arm_travel():
    TTLServo.xyInputSmooth(ARM_TRAVEL_X, ARM_TRAVEL_Y, 0.8)
    time.sleep(1.0)

def arm_lower_for_pickup():
    TTLServo.xyInputSmooth(ARM_PICK_X, ARM_PICK_Y, 0.8)
    time.sleep(1.0)

def arm_lift():
    TTLServo.xyInputSmooth(ARM_LIFT_X, ARM_LIFT_Y, 0.8)
    time.sleep(1.0)

def drop_cap():
    TTLServo.xyInputSmooth(ARM_DROP_X, ARM_DROP_Y, 0.8)
    time.sleep(1.0)
    claw_open()
    time.sleep(0.5)
    arm_lift()

def grab_cap():
    claw_open()
    arm_lower_for_pickup()
    claw_close()
    arm_lift()

arm_travel()
claw_open()

In [ ]:
# CELL 9 — Approach and verification

def choose_nearest_safe_cap(caps, grid):
    safe_caps = []

    for cap in caps:
        gx, gy = world_to_grid(*cap["world"])
        h, w = grid.shape

        if 0 <= gx < w and 0 <= gy < h and grid[gy, gx] == 0:
            d = distance((robot_pose["x"], robot_pose["y"]), cap["world"])
            safe_caps.append((d, cap))

    if not safe_caps:
        return None

    safe_caps.sort(key=lambda x: x[0])
    return safe_caps[0][1]

def verify_cap_not_sphere(frame, target_world):
    caps = detect_bottle_caps(frame)
    spheres = detect_spheres(frame)

    for s in spheres:
        if distance(s["world"], target_world) < 0.12:
            return False

    for c in caps:
        if distance(c["world"], target_world) < 0.12:
            return True

    return False

def approach_object_slowly(target_world):
    approach_point = target_world

    for _ in range(30):
        frame = camera.value
        obstacles = detect_obstacles(frame)

        if not emergency_check(frame, obstacles):
            hard_stop()
            move_backward(duration=0.25)
            return False

        d = distance((robot_pose["x"], robot_pose["y"]), approach_point)

        if d < 0.12:
            hard_stop()
            return True

        rotate_towards(approach_point)
        safe_step_forward(frame, obstacles, CFG["speed_slow"])

    return False

In [ ]:
# CELL 10 — Debug visualization

def draw_detections(frame, obstacles, caps, spheres, grey_box, path=None):
    out = frame.copy()

    # Draw calibrated field
    pts = IMAGE_FIELD_CORNERS.astype(np.int32).reshape((-1, 1, 2))
    cv2.polylines(out, [pts], True, (0, 255, 255), 2)

    for obs in obstacles:
        x, y, w, h = obs["bbox"]
        color = (255, 0, 0) if "blue" in obs["kind"] else (0, 255, 0)
        cv2.rectangle(out, (x, y), (x+w, y+h), color, 2)
        cv2.putText(out, obs["kind"], (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    for cap in caps:
        x, y, w, h = cap["bbox"]
        cv2.rectangle(out, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.putText(out, "CAP", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)

    for sph in spheres:
        x, y, w, h = sph["bbox"]
        cv2.circle(out, sph["center_px"], max(w, h)//2, (255, 0, 255), 2)
        cv2.putText(out, "IGNORE SPHERE", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)

    if grey_box is not None:
        x, y, w, h = grey_box["bbox"]
        cv2.rectangle(out, (x, y), (x+w, y+h), (160, 160, 160), 2)
        cv2.putText(out, "DROP BOX", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (160, 160, 160), 1)

    if path:
        pts = [world_to_image(x, y) for x, y in path]
        for i in range(len(pts)-1):
            cv2.line(out, pts[i], pts[i+1], (255, 255, 255), 2)

    rx, ry = world_to_image(robot_pose["x"], robot_pose["y"])
    cv2.circle(out, (rx, ry), 6, (0, 255, 255), -1)
    cv2.putText(out, "ROBOT", (rx+5, ry),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

    return out

def show_debug(frame, obstacles, caps, spheres, grey_box, path=None):
    vis = draw_detections(frame, obstacles, caps, spheres, grey_box, path)
    debug_image.value = bgr8_to_jpeg(vis)

In [ ]:
# CELL 11 — Main autonomous loop

def autonomous_loop(max_cycles=10):
    hard_stop()
    arm_travel()
    claw_open()

    for cycle in range(max_cycles):
        frame = camera.value

        boundary_mask, boundary_lines = detect_boundary(frame)
        obstacles = detect_obstacles(frame)
        grey_box = detect_grey_box(frame)
        spheres = detect_spheres(frame)
        caps = detect_bottle_caps(frame)

        grid = build_local_map(obstacles)

        show_debug(frame, obstacles, caps, spheres, grey_box)

        if grey_box is None:
            print("Grey drop box not detected. Stopping for safety.")
            hard_stop()
            time.sleep(1)
            continue

        if len(caps) == 0:
            print("No bottle caps detected. Task complete or scan needed.")
            hard_stop()
            break

        target_cap = choose_nearest_safe_cap(caps, grid)

        if target_cap is None:
            print("No safe reachable cap. Replanning after scan.")
            hard_stop()
            time.sleep(1)
            continue

        cap_pos = target_cap["world"]
        drop_pos = grey_box["world"]

        path_to_cap = simplify_path(astar(grid, (robot_pose["x"], robot_pose["y"]), cap_pos))
        show_debug(frame, obstacles, caps, spheres, grey_box, path_to_cap)

        if path_to_cap is None:
            print("No safe path to cap. Skipping.")
            hard_stop()
            time.sleep(1)
            continue

        print(f"Cycle {cycle+1}: navigating to bottle cap at {cap_pos}")
        ok = follow_path(path_to_cap)

        if not ok:
            print("Navigation to cap failed. Replanning.")
            hard_stop()
            continue

        frame = camera.value

        if not verify_cap_not_sphere(frame, cap_pos):
            print("Target is uncertain or sphere-like. Not picking up.")
            hard_stop()
            move_backward(duration=0.25)
            continue

        ok = approach_object_slowly(cap_pos)

        if not ok:
            print("Could not safely approach cap.")
            hard_stop()
            continue

        print("Picking up cap.")
        grab_cap()

        frame = camera.value
        obstacles = detect_obstacles(frame)
        grey_box = detect_grey_box(frame)

        if grey_box is None:
            print("Lost grey box after pickup. Stopping.")
            hard_stop()
            continue

        grid = build_local_map(obstacles)
        drop_pos = grey_box["world"]

        path_to_box = simplify_path(astar(grid, (robot_pose["x"], robot_pose["y"]), drop_pos))
        show_debug(frame, obstacles, [], [], grey_box, path_to_box)

        if path_to_box is None:
            print("No safe path to grey box. Holding object and stopping.")
            hard_stop()
            continue

        print(f"Navigating to grey drop box at {drop_pos}")
        ok = follow_path(path_to_box)

        if not ok:
            print("Navigation to drop box failed. Replanning.")
            hard_stop()
            continue

        print("Dropping cap.")
        drop_cap()
        move_backward(duration=0.25)
        arm_travel()

    hard_stop()
    print("Autonomous loop finished.")

In [ ]:
# CELL 12 — Run this only when the field is clear and calibrated

# autonomous_loop(max_cycles=20)

In [ ]:
# CELL 13 — Emergency stop / cleanup

hard_stop()
camera.stop()

In [ ]:
IMAGE_FIELD_CORNERS
CFG["HSV"]
CLAW_OPEN_ANGLE
CLAW_CLOSED_ANGLE
ARM_PICK_X, ARM_PICK_Y
CFG["obstacle_margin_m"]
CFG["boundary_margin_m"]